# IFC4 Reference View Exporter

This notebook demonstrates how to create building geometry using topologic_fast and prepare it for IFC4 Reference View export.

## Overview

The IFC4 Reference View is a Model View Definition (MVD) that focuses on reference models for coordination purposes. This notebook shows:

1. Creating wall geometry from 2D profiles using topologic_fast
2. Triangulating geometry for mesh-based IFC export
3. Preparing mesh data in a format suitable for IFC tessellation
4. Visualizing the results with Plotly

**Note**: The actual IFC export functionality requires the `ifcopenshell` library and is not yet implemented in topologic_fast. This notebook focuses on geometry preparation.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np

## Helper Functions

First, let's define some helper functions for creating wall geometry and preparing mesh data.

In [ ]:
def WallCellByProfile(profile_xy, height, origin=(0.0, 0.0, 0.0)):
    """
    Create a simple prismatic wall Cell by extruding a closed 2D profile.
    
    Parameters
    ----------
    profile_xy : list[tuple[float, float]]
        Closed loop XY points (you may omit repeating the first point).
    height : float
        Extrusion height along +Z.
    origin : tuple[float, float, float]
        World origin offset for the profile.
    
    Returns
    -------
    Cell
        A prismatic solid (Cell).
    """
    if not profile_xy or len(profile_xy) < 3:
        raise ValueError("profile_xy must have at least 3 points")
    
    # Ensure closed
    pts = list(profile_xy)
    if pts[0] != pts[-1]:
        pts.append(pts[0])
    
    ox, oy, oz = origin
    
    # Create bottom vertices
    bottom_vertices = [tf.Vertex.ByCoordinates(ox + x, oy + y, oz) for (x, y) in pts[:-1]]
    
    # Create wire from vertices
    bottom_wire = tf.Wire.ByVertices(bottom_vertices, True)  # closed=True
    
    # Create bottom face
    bottom_face = tf.Face.ByWire(bottom_wire)
    
    # Create top vertices
    top_vertices = [tf.Vertex.ByCoordinates(ox + x, oy + y, oz + height) for (x, y) in pts[:-1]]
    
    # Create top wire and face
    top_wire = tf.Wire.ByVertices(top_vertices, True)
    top_face = tf.Face.ByWire(top_wire)
    
    # Create side faces
    side_faces = []
    n = len(bottom_vertices)
    for i in range(n):
        v1 = bottom_vertices[i]
        v2 = bottom_vertices[(i + 1) % n]
        v3 = top_vertices[(i + 1) % n]
        v4 = top_vertices[i]
        
        side_wire = tf.Wire.ByVertices([v1, v2, v3, v4], True)
        side_face = tf.Face.ByWire(side_wire)
        side_faces.append(side_face)
    
    # Combine all faces to create cell
    all_faces = [bottom_face, top_face] + side_faces
    cell = tf.Cell.ByFaces(all_faces)
    
    return cell

In [ ]:
def MeshForIFC(cell):
    """
    Convert a Cell to a triangle mesh dict for IFC tessellation.
    
    Returns
    -------
    dict with keys:
      - "vertices": list[(x, y, z)]
      - "faces": list[(i, j, k)]  # 0-based triangle indices
    """
    mesh = tf.Mesh.ByCell(cell)
    
    # NOTE: IFC.ExportReferenceView is not yet implemented in topologic_fast
    # This function demonstrates how mesh data would be extracted
    
    # Get mesh statistics
    num_verts = mesh.NumVertices()
    num_tris = mesh.NumTriangles()
    
    # Export to OBJ format to extract vertex and face data
    obj_content = mesh.ToOBJ()
    
    # Parse OBJ content
    vertices = []
    faces = []
    
    for line in obj_content.strip().split('\n'):
        parts = line.strip().split()
        if not parts:
            continue
        if parts[0] == 'v':
            x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
            vertices.append((x, y, z))
        elif parts[0] == 'f':
            # OBJ uses 1-based indices
            indices = [int(p.split('/')[0]) - 1 for p in parts[1:]]
            if len(indices) == 3:
                faces.append(tuple(indices))
    
    return {"vertices": vertices, "faces": faces}

## Create Wall Geometry

Let's create a wall with a C-shaped profile, similar to an architectural wall with returns.

In [ ]:
# Create a C-shaped profile
c_shape = tf.Wire.CShape()

# Get vertices from the C-shape
c_vertices = c_shape.Vertices()
profile = [(v.X(), v.Y()) for v in c_vertices]

print(f"C-Shape profile has {len(profile)} vertices:")
for i, (x, y) in enumerate(profile):
    print(f"  {i}: ({x:.3f}, {y:.3f})")

In [ ]:
# Create wall cell by extruding the profile
height = 3.0  # Wall height in meters

wall_cell = WallCellByProfile(profile, height)

print(f"Wall Cell created:")
print(f"  Volume: {wall_cell.Volume():.3f} m^3")
print(f"  Surface Area: {wall_cell.Area():.3f} m^2")

In [ ]:
# Generate mesh data for IFC export
wall_mesh = MeshForIFC(wall_cell)

print(f"Wall Mesh:")
print(f"  Vertices: {len(wall_mesh['vertices'])}")
print(f"  Triangles: {len(wall_mesh['faces'])}")

## Prepare IFC Element Payload

This is how you would structure the data for IFC export. Note that the actual IFC export requires the `ifcopenshell` library.

In [ ]:
# Structure the element data for IFC export
elements = [
    {
        "id": "wall_001",
        "ifc_class": "IfcWall",
        "name": "Wall A",
        "tag": "W-001",
        "type_name": "Basic Wall 200mm",
        "dictionary": {"FireRating": "60min"},
        "mesh": wall_mesh
    }
]

# Display the element structure
import json
print("IFC Element Payload (first element):")
element_preview = {k: v for k, v in elements[0].items() if k != 'mesh'}
element_preview['mesh'] = f"{{vertices: {len(wall_mesh['vertices'])}, faces: {len(wall_mesh['faces'])}}}"
print(json.dumps(element_preview, indent=2))

In [ ]:
# NOTE: IFC.ExportReferenceView is not yet implemented in topologic_fast
# The following would be the call in topologicpy:
#
# from topologicpy.IFC import IFC
# IFC.ExportReferenceView(
#     elements,
#     r"/path/to/output/topologic_rv.ifc"
# )

print("IFC Export Note:")
print("  IFC.ExportReferenceView is not yet implemented in topologic_fast.")
print("  To export to IFC, you would need to use topologicpy or ifcopenshell directly.")
print("  ")
print("  The mesh data generated above can be used with ifcopenshell to create")
print("  an IfcTriangulatedFaceSet for the wall geometry.")

## Visualize the Wall Geometry

Let's visualize the wall geometry using Plotly.

In [ ]:
def visualize_mesh(mesh_data, title="Mesh Visualization"):
    """
    Visualize mesh data using Plotly.
    """
    vertices = mesh_data['vertices']
    faces = mesh_data['faces']
    
    # Extract coordinates
    x = [v[0] for v in vertices]
    y = [v[1] for v in vertices]
    z = [v[2] for v in vertices]
    
    # Extract face indices
    i = [f[0] for f in faces]
    j = [f[1] for f in faces]
    k = [f[2] for f in faces]
    
    fig = go.Figure(data=[
        go.Mesh3d(
            x=x, y=y, z=z,
            i=i, j=j, k=k,
            color='lightblue',
            opacity=0.8,
            flatshading=True,
            name='Wall Surface'
        )
    ])
    
    # Add wireframe edges
    for face in faces:
        pts = [vertices[idx] for idx in face] + [vertices[face[0]]]
        fig.add_trace(go.Scatter3d(
            x=[p[0] for p in pts],
            y=[p[1] for p in pts],
            z=[p[2] for p in pts],
            mode='lines',
            line=dict(color='darkblue', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=800,
        height=600
    )
    
    return fig

fig = visualize_mesh(wall_mesh, "C-Shaped Wall - IFC Export Ready")
fig.show()

## Complex Building Example

Let's create a more complex building with multiple walls and decompose it for IFC export.

In [ ]:
# Create a simple building envelope
width = 26
length = 14
height = 9  # Three stories at 3m each

# Main building prism
building = tf.Cell.Prism(
    width=width,
    length=length,
    height=height
)

print(f"Building Envelope:")
print(f"  Dimensions: {width}m x {length}m x {height}m")
print(f"  Volume: {building.Volume():.1f} m^3")
print(f"  Surface Area: {building.Area():.1f} m^2")

In [ ]:
# Generate mesh for the building
building_mesh = MeshForIFC(building)

print(f"Building Mesh:")
print(f"  Vertices: {len(building_mesh['vertices'])}")
print(f"  Triangles: {len(building_mesh['faces'])}")

# Visualize
fig = visualize_mesh(building_mesh, "Building Envelope - IFC Export Ready")
fig.show()

In [ ]:
# Prepare multiple IFC elements
building_elements = [
    {
        "id": "building_001",
        "ifc_class": "IfcBuildingElementProxy",
        "name": "Main Building",
        "tag": "BLDG-001",
        "type_name": "Building Mass",
        "dictionary": {
            "GrossVolume": building.Volume(),
            "GrossArea": building.Area(),
            "NumberOfStoreys": 3
        },
        "mesh": building_mesh
    }
]

print(f"Building Element prepared for IFC export")
print(f"  ID: {building_elements[0]['id']}")
print(f"  Class: {building_elements[0]['ifc_class']}")
print(f"  Properties: {building_elements[0]['dictionary']}")

## Export to OBJ/STL

While IFC export is not yet available, topologic_fast supports exporting to OBJ and STL formats.

In [ ]:
# Export building mesh to OBJ format
mesh = tf.Mesh.ByCell(building)

# Get OBJ content
obj_content = mesh.ToOBJ()
print("OBJ Content (first 500 characters):")
print(obj_content[:500])
print("...")

# Get STL content
stl_content = mesh.ToSTL()
print("\nSTL Content (first 500 characters):")
print(stl_content[:500])
print("...")

In [ ]:
# Save to files (uncomment to save)
# with open('building.obj', 'w') as f:
#     f.write(obj_content)
# print("Saved building.obj")

# with open('building.stl', 'w') as f:
#     f.write(stl_content)
# print("Saved building.stl")

print("To save files, uncomment the code above.")

## Summary

This notebook demonstrated:

1. **Wall Creation**: Creating wall geometry from 2D profiles using topologic_fast
2. **Mesh Generation**: Converting cells to triangle meshes using `tf.Mesh.ByCell()`
3. **IFC Data Preparation**: Structuring mesh data for IFC tessellation export
4. **Visualization**: Using Plotly for 3D visualization of geometry
5. **Export Formats**: OBJ and STL export capabilities

### Not Yet Implemented in topologic_fast

The following topologicpy features are not yet available:

- `IFC.ExportReferenceView()` - IFC4 Reference View export
- `Topology.Decompose()` - Decompose topology into component faces
- `Wire.Ribbon()` - Create ribbon/offset wires
- `Dictionary` integration with topology objects

For IFC export, consider using:
- The original topologicpy library
- ifcopenshell directly with the mesh data generated here